# AutoMorphalyzer on three shapes, biomarker by biomarker

A close reading of one implementation. The
[main notebook](../biomarker-synthetic.ipynb) compares six programs across nine shapes and answers
*which of these agree*; this one asks a narrower question about
[AutoMorphalyzer](../../docs/projects/automorphalyzer.md) alone — **what does it actually return, on the three
simplest shapes, at each angle, beside what the geometry requires?**

Everything is under **AutoMorphalyzer's own column names**, not the catalogued ones. That is the point:
the catalogued names exist to make two programs comparable, and there is only one program here, so
the names its authors chose are the ones a reader checking against their documentation will look
for. Where a column maps onto a catalogued biomarker the mapping is shown beside it, and where the
shape settles a theoretical value that is shown too.

**The three shapes, in order of how much they ask for:**

| Shape | What it is | Why it comes first |
| --- | --- | --- |
| `straight` | one straight vessel per class, leaving the optic disc | every curvature measure must read zero and every tortuosity exactly 1 |
| `arc` | a circular arc per class | curvature is exactly 1/r, and the arc-to-chord ratio has a closed form |
| `sinusoid` | a sine wave per class | the only one of the three whose curvature changes sign |

A blank in the theory column means **this shape settles no value for that quantity** — which is not
the same as the quantity being wrong, and not the same as AutoMorph declining to answer it.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def repository() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(f"nothing above {Path.cwd()} looks like the fundus-atlas repository")


ROOT = repository()
sys.path.insert(0, str(ROOT / "src"))

from benchmarks import biomarker_synthetic as benchmark  # noqa: E402
from benchmarks.shapes import store  # noqa: E402
from biomarkers import canonical  # noqa: E402
from biomarkers.utils import catalogue  # noqa: E402

#: The one implementation this notebook reads. Change it and everything below follows.
IMPLEMENTATION = "automorphalyzer"

#: In order of how much they ask for: a straight vessel, then a curve, then a curve that changes
#: its mind about which way it bends.
SHAPES = ("straight", "arc", "sinusoid")

EVIDENCE = ROOT / "results" / benchmark.NAME / IMPLEMENTATION
STORE = ROOT / store.STORE
pd.set_option("display.width", 200, "display.max_rows", 200)

#: AutoMorph's own column name → the catalogued biomarker it answers to, or None where the
#: catalogue has no name for it. This is the adapter's claim, not the run's.
NAMES = catalogue.load(IMPLEMENTATION).declare()["names"]

TRUTH = pd.read_csv(STORE / "ground_truth.csv").set_index("key")

#: How many microns a pixel is worth, read from the manifest rather than assumed.
UM_PER_PX = float(pd.DataFrame(store.read(STORE))["um_per_px"].iloc[0])
print(f"{IMPLEMENTATION}: {len(NAMES)} columns, "
      f"{sum(1 for v in NAMES.values() if v)} of them with a catalogued name")

automorphalyzer: 54 columns, 23 of them with a catalogued name


In [2]:
def readings(shape: str) -> pd.DataFrame:
    """What AutoMorph returned for one shape: a row per column, a column per angle.

    The theoretical value is joined on last and only where the shape settles one, so a blank there
    **Every number in a row is in the implementation's own units, which are pixels.** A catalogued
    biomarker is in microns, so the theory is converted *back* onto the pixel grid before it is put
    beside figures this program actually printed. The `unit` column says what the catalogued name
    is in.

    The theoretical value is joined on last, so a blank is a statement about the fixture rather
    than about AutoMorph.
    """
    measured = pd.read_csv(EVIDENCE / f"{shape}.csv")
    rows = {}
    for own in NAMES:
        values = {}
        for _, rendering in measured.iterrows():
            values[f"{rendering['rotation']:.0f}°"] = rendering.get(f"said_{own}")
        catalogued = NAMES[own]
        truth, unit = np.nan, ""
        if catalogued:
            unit = canonical.unit(catalogued)
            keys = [k for k in TRUTH.index if k.startswith(f"{shape}-")]
            if keys and catalogued in TRUTH.columns:
                # Back onto the pixel grid: the inverse of the one conversion this repository makes.
                truth = canonical.from_pixels(
                    catalogued, TRUTH.loc[keys[0], catalogued], 1.0 / UM_PER_PX
                )
        rows[own] = {
            **values,
            "theory, in px": truth,
            "catalogued as": catalogued or "—",
            "unit": unit or "—",
        }
    frame = pd.DataFrame(rows).T
    frame.index.name = f"{IMPLEMENTATION}'s own name"
    return frame


def show(shape: str) -> pd.DataFrame:
    frame = readings(shape)
    angles = [c for c in frame.columns if c.endswith("°")]
    numeric = frame[angles + ["theory, in px"]].astype(float)
    spread = numeric[angles].max(axis=1) - numeric[angles].min(axis=1)
    out = numeric.round(4)
    out["moves"] = spread.round(4)
    out["catalogued as"] = frame["catalogued as"]
    out["unit"] = frame["unit"]
    return out

## 1. `straight`

A single straight vessel per class, leaving the optic disc. **Everything about it is known
exactly**: the arc-to-chord ratio is 1, every curvature measure is 0, and the width is the width it
was drawn at. Nothing here is a hard measurement, which is what makes it the right place to start —
a value that is wrong on a straight line is wrong for a reason that has nothing to do with
difficulty.

In [3]:
show("straight")

,0°,30°,60°,90°,"theory, in px",moves,catalogued as,unit
automorphalyzer's own name,,,,,,,,
fractal_dimension@whole_binary,0.8971,0.9996,1.0112,0.8971,NaN,0.1141,density/box-counting/vessels,1
vessel_density@whole_binary,0.0059,0.0059,0.0059,0.0059,0.0060,0.0000,density/over-image/vessels,1
average_global_calibre@whole_binary,20.5665,22.7046,22.8059,20.5835,20.0000,2.2394,calibre/width/vessels/length-weighted,µm
tortuosity_distance@whole_binary,1.0007,1.0709,1.0719,1.0007,1.0000,0.0712,tortuosity/hart-tau1/vessels/mean,1
tortuosity_density@whole_binary,0.0000,0.4578,0.4578,0.2500,0.0000,0.4578,tortuosity/grisan-density/vessels/mean,1/µm
average_local_calibre@whole_binary,20.0105,17.9221,18.4420,20.0013,NaN,2.0884,—,—
CRAE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRVE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRAE_Knudtson@B_binary,NaN,NaN,NaN,NaN,NaN,NaN,—,—


## 2. `arc`

A circular arc per class, the two sharing a subtended angle and differing in radius. The
arc-to-chord ratio is `θ / (2 sin(θ/2))` in closed form and **independent of radius**, so both
classes must return the same tortuosity while their curvatures differ. That pairing is the test:
an implementation that returns different tortuosities for the two is measuring the radius somewhere
it should not.

In [4]:
show("arc")

,0°,30°,60°,90°,"theory, in px",moves,catalogued as,unit
automorphalyzer's own name,,,,,,,,
fractal_dimension@whole_binary,1.0173,1.0229,1.0696,1.0173,NaN,0.0522,density/box-counting/vessels,1
vessel_density@whole_binary,0.0058,0.0058,0.0058,0.0058,0.0059,0.0000,density/over-image/vessels,1
average_global_calibre@whole_binary,21.8335,21.4424,21.2298,21.8140,19.4286,0.6037,calibre/width/vessels/length-weighted,µm
tortuosity_distance@whole_binary,1.1381,1.1160,1.0976,1.1364,1.1107,0.0405,tortuosity/hart-tau1/vessels/mean,1
tortuosity_density@whole_binary,0.6625,0.6842,0.4991,0.9382,0.0000,0.4392,tortuosity/grisan-density/vessels/mean,1/µm
average_local_calibre@whole_binary,20.2628,17.7410,19.0904,20.3249,NaN,2.5839,—,—
CRAE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRVE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRAE_Knudtson@B_binary,NaN,NaN,NaN,NaN,NaN,NaN,—,—


## 3. `sinusoid`

A sine wave per class, and **the only one of the three whose curvature changes sign**. Its arc
length is an elliptic integral with no elementary closed form, so the theoretical value here is the
integral rather than a formula. It is the shape that settles the inflection count and Grisan's
density, both of which are zero on everything else.

In [5]:
show("sinusoid")

,0°,30°,60°,90°,"theory, in px",moves,catalogued as,unit
automorphalyzer's own name,,,,,,,,
fractal_dimension@whole_binary,1.0266,1.0071,1.0462,1.0266,NaN,0.0391,density/box-counting/vessels,1
vessel_density@whole_binary,0.0065,0.0065,0.0065,0.0065,0.0066,0.0000,density/over-image/vessels,1
average_global_calibre@whole_binary,24.0547,20.9846,21.8114,24.0547,20.0000,3.0701,calibre/width/vessels/length-weighted,µm
tortuosity_distance@whole_binary,1.4528,1.4635,1.4766,1.4532,1.3726,0.0238,tortuosity/hart-tau1/vessels/mean,1
tortuosity_density@whole_binary,0.9930,0.9922,0.9950,0.9929,0.0011,0.0028,tortuosity/grisan-density/vessels/mean,1/µm
average_local_calibre@whole_binary,20.5778,20.3739,20.4793,20.4651,NaN,0.2040,—,—
CRAE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRVE_Knudtson@whole_binary,NaN,NaN,NaN,NaN,NaN,NaN,—,—
CRAE_Knudtson@B_binary,NaN,NaN,NaN,NaN,NaN,NaN,—,—


## 4. What the three tables show

AutoMorphalyzer returns 54 columns, three times what its ancestor
[AutoMorph](biomarker-synthetic-automorph.ipynb) does, because it reports most quantities once per
**measurement zone** — `@whole`, `@B` and `@C`. Zones B and C are the conventional annuli round the
optic disc from the ARIC literature. Having three of everything is what makes these tables long and
also what makes them informative: the same quantity measured three ways is a check nothing else
here offers.

### 4.1 Twelve columns that never answer

Every `CRAE_Knudtson` and `CRVE_Knudtson` column — twelve of them, across three zones and both
classes — is **empty on all three shapes at all four angles**. The central retinal equivalents
combine the widths of several vessels crossing a ring round the disc, and none of these three
shapes puts several vessels across that ring: `straight`, `arc` and `sinusoid` each draw one vessel
per class.

That is the fixture's doing rather than a defect, and it is worth saying because an empty column
looks the same as a broken one. The shapes that would exercise these are the two `spokes` families,
which draw twelve vessels through the ring on purpose.

### 4.2 Two calibres that disagree with each other by nine pixels

It reports both a `average_global_calibre` and an `average_local_calibre` for the same vessel:

| on `straight`, vein | 0° | 30° | 60° | 90° | drawn at |
| --- | --- | --- | --- | --- | --- |
| `average_global_calibre@whole_vein` | 24.91 | 27.13 | 27.23 | 24.91 | 24 |
| `average_local_calibre@whole_vein` | **24.02** | **18.33** | **19.02** | **24.00** | 24 |

At 0° and 90° the local calibre is almost exactly right — 24.02 and 24.00 against a drawn 24 — and
the global is 4% wide. At 30° and 60° they diverge by **more than eight pixels**, a third of the
vessel's width, and the local one is now 24% narrow while the global is 13% wide.

Two numbers for one vessel, both under the implementation's own names, disagreeing by a third of
the thing being measured. Nothing in the catalogue distinguishes them, which is why only one of the
two is mapped and the other is reported unnamed.

### 4.3 Grisan density reads 0.998 on a straight vessel, and the zones disagree

`tortuosity_density` must be **exactly zero** on `straight`: it counts constant-sign bends, and a
straight line has none.

| on `straight`, artery | 0° | 30° | 60° | 90° | theory |
| --- | --- | --- | --- | --- | --- |
| `tortuosity_density@whole_artery` | 0.0000 | 0.0000 | **0.9977** | 0.5000 | 0 |
| `tortuosity_density@B_artery` | 0.0000 | 0.0000 | **0.9924** | 0.0000 | 0 |
| `tortuosity_density@C_artery` | 0.0000 | 0.0000 | **0.9968** | 0.0000 | 0 |

Three zones of the same straight vessel, and at 60° all three report close to 1 for a quantity that
must be 0. The vein does the same thing at a different angle — 0.9923 at 30° in zone B, 0.0000 at
60° — so it is not one angle being unlucky.

Reading the three zones together is what makes this legible: they agree with each other closely at
every angle, which says the zone machinery is working and the quantity being fed through it is not.

### 4.4 A straight line with a fractal dimension of 0.376

`fractal_dimension@whole_artery` on `straight` reads **0.3758, 0.8841, 0.8841, 0.3758**, at the two
axis-aligned angles and the two diagonal ones respectively. A straight line has dimension 1.

Its sibling [AutoMorphClass](biomarker-synthetic-automorphclass.ipynb) returns 0.3758 on the same
shape at the same angles, and their shared ancestor AutoMorph returns 0.927. Both rewrites
introduced it and neither inherited it, which places it in whatever the two changed in common.

### 4.5 What is not checked here

Of 54 columns, 17 carry a catalogued name and the rest are reported unnamed — every zone variant
beyond `@whole`, both calibre measures beyond the first, and every Knudtson equivalent. Unnamed
does not mean wrong; it means the catalogue has no second implementation to compare them against,
so nothing in this notebook can check them.
